In [1]:
import sys, platform
ON_LINUXONE = platform.machine() == "s390x"
DATA_PATH = ("/home/ibm-user/transactions_history.csv" if ON_LINUXONE
             else "transactions_history.csv")
SEED = 42
TRAIN_END_YEAR = 2015         # train: up to 2015
VALID_YEARS = (2016, 2017)    # validation: compare models here
TEST_YEARS = (2018, 2019)     # test: use once in step 22

import pandas as pd
import numpy as np
import sklearn
import xgboost
import tensorflow as tf
import keras

print("Machine:", platform.machine(), "| Python:", sys.version.split()[0])

Machine: s390x | Python: 3.12.15


In [2]:
# 1. Load data
df = pd.read_csv(DATA_PATH)

# 2. Clean Amount: "$134.09" -> 134.09
df['Amount_num'] = df['Amount'].astype(str).str.replace('$', '', regex=False).astype(float)

# 3. Clean 'Is Fraud?' -> 1 / 0
# Bypass the object dtype check to support Pandas 3.x StringDtype
df['is_fraud'] = (df['Is Fraud?'].astype(str).str.strip().str.lower() == 'yes').astype(int)

# 4. Create single timestamp 'ts'
time_split = df['Time'].astype(str).str.split(':', expand=True)
df['hour'] = time_split[0].astype(int)
df['minute'] = time_split[1].astype(int)

df['ts'] = pd.to_datetime(dict(
    year=df['Year'],
    month=df['Month'],
    day=df['Day'],
    hour=df['hour'],
    minute=df['minute']
))

# 5. Sort chronologically
df = df.sort_values(by=['User', 'Card', 'ts']).reset_index(drop=True)

# 6. Assign train / valid / test splits by year
def assign_split(year):
    if year <= TRAIN_END_YEAR:
        return 'train'
    elif year in VALID_YEARS:
        return 'valid'
    elif year in TEST_YEARS:
        return 'test'
    else:
        return 'drop'

df['split'] = df['Year'].apply(assign_split)

# 7. Drop 2020+
df = df[df['split'] != 'drop'].reset_index(drop=True)

# 8. Print fraud counts to verify
fraud_counts = df.groupby('split')['is_fraud'].sum()
print("--- Fraud Count per Split ---")
print(fraud_counts)

--- Fraud Count per Split ---
split
test      60
train    178
valid     33
Name: is_fraud, dtype: int64


In [3]:
from sklearn.metrics import precision_score, recall_score, average_precision_score, confusion_matrix
import numpy as np

def threshold_for_budget(scores, alerts_per_1000=5):
    return float(np.quantile(np.asarray(scores), 1 - alerts_per_1000 / 1000.0))

def report(y_true, scores, threshold=None, alerts_per_1000=5):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    if threshold is None:
        threshold = threshold_for_budget(scores, alerts_per_1000)
    preds = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()

    res = {
        "threshold": threshold,
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "pr_auc": average_precision_score(y_true, scores),
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        "alerts_per_1000": ((tp + fp) / len(y_true)) * 1000
    }
    print(f"--- Evaluation at Budget ({alerts_per_1000} alerts/1000) ---")
    for k, v in res.items():
        print(f"  {k:16s}: {v:.6f}")
    return res

In [4]:
# 1. Base numerical and time features
df['amount'] = df['Amount_num']
df['weekday'] = df['ts'].dt.dayofweek
df['is_refund'] = (df['amount'] < 0).astype(int)

# 2. Chip / Swipe / Online (One-hot encode)
chip_dummies = pd.get_dummies(df['Use Chip'], prefix='tx_type', dtype=int)
df = pd.concat([df, chip_dummies], axis=1)

# 3. Foreign Merchant (US states are 2 letters; everything else is foreign)
US_STATES = set(
    "AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO "
    "MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC".split()
)
state = df['Merchant State']
df['foreign_merchant'] = (state.notna() & ~state.isin(US_STATES)).astype(int)

# 4. Errors (One-hot encode comma-separated error types)
df['Errors?'] = df['Errors?'].fillna('')
errors_dummies = df['Errors?'].str.get_dummies(sep=',').add_prefix('err_')
df = pd.concat([df, errors_dummies], axis=1)

# 5. Define the final feature list
FEATURE_COLUMNS = ['amount', 'hour', 'weekday', 'is_refund', 'foreign_merchant', 'MCC'] + \
                  list(chip_dummies.columns) + list(errors_dummies.columns)

print("--- Feature Columns ---")
for col in FEATURE_COLUMNS:
    print(col)

print("\nFeature table shape:", df[FEATURE_COLUMNS].shape)

--- Feature Columns ---
amount
hour
weekday
is_refund
foreign_merchant
MCC
tx_type_Chip Transaction
tx_type_Online Transaction
tx_type_Swipe Transaction
err_Bad CVV
err_Bad Card Number
err_Bad Expiration
err_Bad PIN
err_Bad Zipcode
err_Insufficient Balance
err_Technical Glitch

Feature table shape: (242493, 16)


In [5]:
# FIX: Added 'amount' to tie-break simultaneous timestamps
df = df.sort_values(by=['User', 'Card', 'ts', 'amount']).reset_index(drop=True)

df_time = df.set_index('ts')
grouped_time = df_time.groupby(['User', 'Card'])

# 1. Velocity (FIX: Subtract 1 from counts, subtract current amount from sums)
df['count_1h'] = grouped_time['amount'].transform(lambda x: x.rolling('1h').count()).reset_index(drop=True) - 1
df['count_24h'] = grouped_time['amount'].transform(lambda x: x.rolling('24h').count()).reset_index(drop=True) - 1
df['count_7d'] = grouped_time['amount'].transform(lambda x: x.rolling('7D').count()).reset_index(drop=True) - 1 # Fixed 7D warning
df['amt_24h'] = grouped_time['amount'].transform(lambda x: x.rolling('24h').sum()).reset_index(drop=True) - df['amount']

# 2. Minutes since previous (Fixed: fill first transaction with 1 year in minutes)
df['mins_since_last'] = df.groupby(['User', 'Card'])['ts'].diff().dt.total_seconds() / 60.0
df['is_first_tx'] = df.groupby(['User', 'Card']).cumcount().eq(0).astype(int)
df['mins_since_last'] = df['mins_since_last'].fillna(365 * 24 * 60)

# 3. Behavior (FIX: Shift median backward by 1 so it only looks at the past)
df['median_so_far'] = df.groupby(['User', 'Card'])['amount'].transform(lambda x: x.shift().expanding().median())
df['median_so_far'] = df['median_so_far'].fillna(0)
df['ratio_median'] = np.where(df['median_so_far'] > 0, df['amount'] / df['median_so_far'], 0)

df['first_time_merchant'] = (~df.duplicated(subset=['User', 'Card', 'Merchant Name'])).astype(int)
df['first_time_city'] = (~df.duplicated(subset=['User', 'Card', 'Merchant City'])).astype(int)

# Update feature list
NEW_FEATURES = ['count_1h', 'count_24h', 'count_7d', 'amt_24h', 'mins_since_last', 
                'ratio_median', 'first_time_merchant', 'first_time_city']
FEATURE_COLUMNS.extend([f for f in NEW_FEATURES if f not in FEATURE_COLUMNS])

print("--- Running Strict Leakage Check ---")
card_df = df[(df['User'] == df['User'].iloc[0]) & (df['Card'] == df['Card'].iloc[0])].copy()
half_idx = len(card_df) // 2

# Check if first half median matches the full calculation perfectly
first_half_median = card_df.iloc[:half_idx]['amount'].shift().expanding().median().fillna(0)
diff = (first_half_median - card_df['median_so_far'].iloc[:half_idx]).abs().max()

if diff == 0:
    print("Leakage check PASSED! (Rolling features are strictly backward-looking)")
else:
    print(f"Leakage check FAILED! Difference: {diff}")

--- Running Strict Leakage Check ---
Leakage check PASSED! (Rolling features are strictly backward-looking)


In [6]:
def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))

for c in ['amount', 'amt_24h', 'ratio_median', 'mins_since_last']:
    df[c + '_log'] = signed_log(df[c])

top_mccs = df.loc[df['split'] == 'train', 'MCC'].value_counts().index[:30]
mcc_oh = pd.get_dummies(df['MCC'].where(df['MCC'].isin(top_mccs), -1), prefix='mcc', dtype=int)
df = pd.concat([df, mcc_oh], axis=1)

NN_FEATURES = [c for c in FEATURE_COLUMNS if c not in ('MCC', 'amount', 'amt_24h', 'ratio_median', 'mins_since_last')] + \
              ['amount_log', 'amt_24h_log', 'ratio_median_log', 'mins_since_last_log'] + \
              list(mcc_oh.columns)

In [7]:
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow.keras as keras
import tensorflow.keras.backend as K
import tensorflow as tf

keras.utils.set_random_seed(SEED)

# 1. Isolate training and validation sets using NN_FEATURES
train_df = df[df['split'] == 'train'].copy()
valid_df = df[df['split'] == 'valid'].copy()

X_train = train_df[NN_FEATURES].copy()
y_train = train_df['is_fraud']
X_valid = valid_df[NN_FEATURES].copy()
y_valid = valid_df['is_fraud']

# Calculate class weights
num_non_fraud = (y_train == 0).sum()
num_fraud = (y_train == 1).sum()
total = num_non_fraud + num_fraud
class_weight = {0: (1 / num_non_fraud) * (total / 2.0), 1: (1 / num_fraud) * (total / 2.0)}

# 2. Scale and clip features [-10, 10]
scaler = StandardScaler()
X_train_scaled = np.clip(scaler.fit_transform(X_train), -10, 10)
X_valid_scaled = np.clip(scaler.transform(X_valid), -10, 10)

# 3. Build Model using Input layer
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(curve='PR', name='pr_auc')]
)

# 4. LinuxONE Workaround Callbacks
class FreshValidationGraph(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.model.test_function = None

def predict_fresh(m, X, batch_size=4096):
    m.make_predict_function(force=True)
    return m.predict(X, batch_size=batch_size, verbose=0).ravel()

early_stopping = EarlyStopping(monitor='val_pr_auc', patience=5, mode='max', restore_best_weights=True)

print("\nTraining TensorFlow model...")
model.fit(
    X_train_scaled, y_train,
    epochs=30,
    batch_size=2048,
    validation_data=(X_valid_scaled, y_valid),
    class_weight=class_weight,
    callbacks=[FreshValidationGraph(), early_stopping],
    verbose=1
)

valid_scores_tf = predict_fresh(model, X_valid_scaled)
print("\n--- TensorFlow Validation Results ---")
tf_metrics = report(y_valid, valid_scores_tf, alerts_per_1000=5)


Training TensorFlow model...
Epoch 1/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.5994 - pr_auc: 0.0066 - val_loss: 0.5342 - val_pr_auc: 0.0161
Epoch 2/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3863 - pr_auc: 0.0197 - val_loss: 0.2832 - val_pr_auc: 0.0230
Epoch 3/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.2617 - pr_auc: 0.0398 - val_loss: 0.1696 - val_pr_auc: 0.0250
Epoch 4/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.2210 - pr_auc: 0.0375 - val_loss: 0.1480 - val_pr_auc: 0.0281
Epoch 5/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1806 - pr_auc: 0.0464 - val_loss: 0.1135 - val_pr_auc: 0.0302
Epoch 6/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1708 - pr_auc: 0.0633 - val_loss: 0.1148 - val_pr_auc: 0.0302
Epoch 7/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1483 - pr_auc: 0.0721 - val_loss: 0.0981 - val_pr_auc: 0.0303
Epoch 8/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1600 - pr_auc: 0.0744 - val_loss: 0.0904 - val_pr_auc: 0

In [8]:
from tensorflow.keras.layers import LSTM, Masking, Input
import numpy as np

SEQ_LEN = 10

def create_sequences_all(scaled, seq_len=SEQ_LEN):
    X = np.zeros((len(df), seq_len, scaled.shape[1]), dtype='float32')
    rows = np.arange(len(df))
    pos = df.groupby(['User', 'Card']).cumcount().values
    for lag in range(seq_len):
        ok = pos >= lag
        if ok.any():
            X[ok, seq_len - 1 - lag] = scaled[rows[ok] - lag]
    return X

print("Building global sequences for LSTM...")
scaled_nn_all = np.clip(scaler.transform(df[NN_FEATURES]), -10, 10)
X_all_seq = create_sequences_all(scaled_nn_all)

train_mask = (df['split'] == 'train').values
valid_mask = (df['split'] == 'valid').values

X_train_seq = X_all_seq[train_mask]
y_train_seq = df.loc[train_mask, 'is_fraud'].values
X_valid_seq = X_all_seq[valid_mask]
y_valid_seq = df.loc[valid_mask, 'is_fraud'].values

# Build LSTM Model
lstm_model = Sequential([
    Input(shape=(SEQ_LEN, len(NN_FEATURES))),
    Masking(mask_value=0.),
    LSTM(32, activation='tanh', return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(curve='PR', name='pr_auc')]
)

print("\nTraining LSTM model...")
lstm_model.fit(
    X_train_seq, y_train_seq,
    epochs=15,
    batch_size=1024,
    validation_data=(X_valid_seq, y_valid_seq),
    class_weight=class_weight,
    callbacks=[FreshValidationGraph(), EarlyStopping(monitor='val_pr_auc', patience=4, mode='max', restore_best_weights=True)],
    verbose=1
)

valid_scores_lstm = predict_fresh(lstm_model, X_valid_seq)
print("\n--- LSTM Sequence Validation Results ---")
lstm_metrics = report(y_valid_seq, valid_scores_lstm, alerts_per_1000=5)

Building global sequences for LSTM...

Training LSTM model...
Epoch 1/15


2026-10-17 11:48:58.512549: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 378639360 exceeds 10% of free system memory.


172/172 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 0.4315 - pr_auc: 0.1635 - val_loss: 0.3495 - val_pr_auc: 0.0204
Epoch 2/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.1481 - pr_auc: 0.4856 - val_loss: 0.1126 - val_pr_auc: 0.0190
Epoch 3/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0775 - pr_auc: 0.5787 - val_loss: 0.0627 - val_pr_auc: 0.0182
Epoch 4/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0571 - pr_auc: 0.5644 - val_loss: 0.0572 - val_pr_auc: 0.0170
Epoch 5/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0329 - pr_auc: 0.5675 - val_loss: 0.0223 - val_pr_auc: 0.0134

--- LSTM Sequence Validation Results ---
--- Evaluation at Budget (5 alerts/1000) ---
  threshold       : 0.870117
  precision       : 0.022989
  recall          : 0.121212
  pr_auc          : 0.025735
  fpr             : 0.005056
  alerts_per_1000 : 5.170262


In [9]:
import xgboost as xgb

# Re-isolate data
train_df = df[df['split'] == 'train'].copy()
valid_df = df[df['split'] == 'valid'].copy()
X_train = train_df[FEATURE_COLUMNS]
y_train = train_df['is_fraud']
X_valid = valid_df[FEATURE_COLUMNS]
y_valid = valid_df['is_fraud']

# Recalculate weights and train
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=scale_weight,
    random_state=SEED,
    eval_metric='aucpr'
)

print("Restoring XGBoost model...")
xgb_model.fit(X_train, y_train)
valid_scores_xgb = xgb_model.predict_proba(X_valid)[:, 1]

print("--- XGBoost Restored ---")
_ = report(y_valid, valid_scores_xgb)  # uses the standard 5-per-1,000 alert budget

Restoring XGBoost model...
--- XGBoost Restored ---
--- Evaluation at Budget (5 alerts/1000) ---
  threshold       : 0.007412
  precision       : 0.076923
  recall          : 0.393939
  pr_auc          : 0.335891
  fpr             : 0.004640
  alerts_per_1000 : 5.021691


In [10]:
from sklearn.metrics import average_precision_score

# 1. Gather predictions using predict_fresh
xgb_probs = xgb_model.predict_proba(X_valid)[:, 1]
tf_probs = predict_fresh(model, X_valid_scaled)
lstm_probs = predict_fresh(lstm_model, X_valid_seq)

# 2. Grid search best ensemble weights on validation via Average Precision
best_w = max(
    ((w1, w2, round(1 - w1 - w2, 2)) 
     for w1 in np.arange(0, 1.0001, 0.05) 
     for w2 in np.arange(0, 1.0001 - w1, 0.05)),
    key=lambda w: average_precision_score(y_valid, w[0]*xgb_probs + w[1]*lstm_probs + w[2]*tf_probs)
)
print("Best weights (XGB, LSTM, TF):", best_w)

ensemble_df = valid_df[['User', 'Card', 'ts', 'Amount_num']].copy()
blend = best_w[0]*xgb_probs + best_w[1]*lstm_probs + best_w[2]*tf_probs
ensemble_df['Blended_Risk'] = blend

# 3. Budget-driven Tiered Routing (Top 1 per 1,000 for RED, Top 5 per 1,000 for YELLOW)
red_thr = threshold_for_budget(blend, alerts_per_1000=1)
yellow_thr = threshold_for_budget(blend, alerts_per_1000=5)

ensemble_df['Action_Tier'] = np.select(
    [blend >= red_thr, blend >= yellow_thr],
    ['RED', 'YELLOW'],
    default='CLEAR'
)
ensemble_df['is_fraud'] = valid_df['is_fraud'].values

print("\n--- 🛡️ Validation Routing Summary by Alert Budget ---")
print(ensemble_df.groupby('Action_Tier')['is_fraud'].agg(transactions='size', frauds='sum'))
print("AP blend:", average_precision_score(y_valid, blend), "| AP XGBoost alone:", average_precision_score(y_valid, xgb_probs))

Best weights (XGB, LSTM, TF): (np.float64(0.7000000000000001), np.float64(0.25), np.float64(0.05))

--- 🛡️ Validation Routing Summary by Alert Budget ---
             transactions  frauds
Action_Tier                      
CLEAR               33485      19
RED                    34      13
YELLOW                135       1
AP blend: 0.36424275861078687 | AP XGBoost alone: 0.33589064126184487
